# Ship-motion benchmark on Kaggle

Run this notebook with **Accelerator: GPU**. It clones the public repository, retains the supplied raw CSVs and immutable M0--M2 manifests, then runs the leakage tests and the M3--M7 benchmark. Generated results are written under `/kaggle/working`.

## Resumable artifacts

Create or attach the private Kaggle Dataset `kushchaudhari/ship-motion-reimplementation-artifacts` as an input before executing. The setup cell restores its contents. Each stage is reused only after its required files and immutable M0--M2 hashes verify; otherwise only that incomplete stage is run. At the end, save a notebook version and update the artifact dataset with the generated `ship-motion-artifacts` directory.

The M8--M12 cell is deliberately opt-in: it is longer-running and should be used only after the primary benchmark has completed and its artifacts are frozen.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kushc2004/ship-motion-reimplementation.git'
WORKDIR = Path('/kaggle/working')
PROJECT_ROOT = WORKDIR / 'ship-motion-reimplementation'

if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
print('Project root:', Path.cwd())
print('Python:', sys.version.split()[0])
!nvidia-smi || true

In [ ]:
# Kaggle images normally include these packages. Install missing dependencies
# from the repository list; enable Internet in Kaggle only if installation is needed.
!{sys.executable} -m pip install -q -r requirements-kaggle.txt

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running neural models.'
print('CUDA device:', torch.cuda.get_device_name(0))

In [ ]:
# Confirm that the exact supplied sources and immutable artifacts are present.
required = [
    Path('BTP-1/Data.csv'),
    Path('BTP-1/Ship-Parameters-data.csv'),
    Path('BTP-2/BTP_2_data - Sheet1.csv'),
    Path('data/manifests/run_manifest.parquet'),
    Path('data/splits/BTP-1_windows.parquet'),
    Path('data/splits/BTP-2_windows.parquet'),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing repository inputs: {missing}')
print('Immutable M0--M2 inputs are present.')

In [ ]:
# Restore a prior artifact dataset if it was attached through Kaggle's Add Input UI.
# This never replaces source data or M0--M2 manifests in the repository.
import shutil

ARTIFACT_DATASET_SLUG = 'ship-motion-reimplementation-artifacts'
ARTIFACT_INPUT = Path('/kaggle/input') / ARTIFACT_DATASET_SLUG
ARTIFACT_ROOT = WORKDIR / 'ship-motion-artifacts'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
if ARTIFACT_INPUT.is_dir():
    shutil.copytree(ARTIFACT_INPUT, ARTIFACT_ROOT, dirs_exist_ok=True)
    print('Restored prior artifacts from:', ARTIFACT_INPUT)
else:
    print('No artifact dataset attached; this is a fresh artifact store.')

shutil.copy2(PROJECT_ROOT / 'kaggle_artifact_store/dataset-metadata.json', ARTIFACT_ROOT / 'dataset-metadata.json')
os.environ['SHIPMOTION_OUTPUT_ROOT'] = str(ARTIFACT_ROOT)
print('Artifact root:', ARTIFACT_ROOT)
!python scripts/check_cached_stage.py primary || true


In [ ]:
# Run the hard leakage and boundary tests before training.
!pytest -q tests/test_m02.py tests/test_m37.py

In [ ]:
# M3--M7: persistence, linear trend, Ridge, RF, XGBoost, LSTM, Transformer.
# The runner reuses a verified cached stage; pass --force only to rerun deliberately.
!python scripts/run_primary_benchmark.py

In [ ]:
# Inspect the primary frozen-test comparison after the previous cell completes.
import pandas as pd
comparison_path = ARTIFACT_ROOT / 'primary_benchmark_m37/primary_comparison.csv'
display(pd.read_csv(comparison_path))

## Optional: M8--M12

Only run this after the preceding primary output has been reviewed and frozen. It consumes the existing immutable manifests and does not rebuild them. M12 export remains gated on completion of the required M10 and M11 artifacts.

In [ ]:
# Optional long-running later-phase experiments. The runner also reuses a verified cache.
# !python scripts/run_m8_m12.py
# !pytest -q tests/test_m812.py
# !python scripts/export_final_report.py
# !python scripts/export_cv_claims.py

# Persist ARTIFACT_ROOT by saving the Kaggle notebook version, then upload/update
# this directory as the attached artifact dataset. Never upload raw inputs or M0--M2 manifests here.
# If this Kaggle session has CLI credentials, uncomment the next line to version it directly:
# !kaggle datasets version -p "$ARTIFACT_ROOT" --message "ship-motion artifact update"
!du -sh "$ARTIFACT_ROOT"
!find "$ARTIFACT_ROOT" -maxdepth 2 -type f | sort